# Ứng dụng học máy trong phân tích hành vi tương tác học tập trực tuyến và dự đoán kết quả học viên

**Nghiên cứu trên OULAD (đầy đủ) + Kiểm chứng chéo trên 3 bộ dữ liệu độc lập ở 4 khu vực địa lý**

Notebook này tổng hợp toàn bộ mã nguồn của nghiên cứu, gồm 4 pipeline:

1. **OULAD** (Anh, 32.593 SV) — pipeline chính, 5 bước
2. **Dropout Academic Success** (Bồ Đào Nha, 2021, 4.424 SV) — đối chứng #1
3. **North America Course** (Bắc Mỹ, 2020, 486 SV) — đối chứng #2
4. **xAPI-Edu** (Trung Đông, 2016, 480 SV) — đối chứng #3

---

## Mục lục

- [0. Cài đặt thư viện](#setup)
- [Phần 1 — Pipeline OULAD (chính)](#part1)
  - [1.1 Tiền xử lý & Feature Engineering](#p1-1)
  - [1.2 Phân tích khám phá dữ liệu (EDA)](#p1-2)
  - [1.3 Huấn luyện & đánh giá 4 mô hình × 3 mốc](#p1-3)
  - [1.4 Phân tích SHAP](#p1-4)
  - [1.5 Biểu đồ bổ sung](#p1-5)
- [Phần 2 — Đối chứng #1: Bồ Đào Nha (2021)](#part2)
- [Phần 3 — Đối chứng #2: Bắc Mỹ (2020)](#part3)
- [Phần 4 — Đối chứng #3: Trung Đông (xAPI-Edu)](#part4)


<a id="setup"></a>
## 0. Cài đặt thư viện

Chạy ô này một lần để cài đặt các thư viện cần thiết.

In [ ]:
# Cài đặt các thư viện (bỏ comment nếu chạy lần đầu)
# !pip install pandas numpy scikit-learn xgboost shap imbalanced-learn matplotlib seaborn joblib pyreadr

import warnings
warnings.filterwarnings("ignore")
print("Sẵn sàng. Đảm bảo đã cài đủ: pandas, numpy, scikit-learn, xgboost, shap, imbalanced-learn, matplotlib, seaborn, joblib, pyreadr")

<a id="part1"></a>
# Phần 1 — Pipeline OULAD (chính)

**Nguồn dữ liệu:** Bộ dữ liệu đầy đủ (32.593 sinh viên, 10.655.280 bản ghi tương tác VLE) có thể tải từ:
- Repo R package chính thức của tác giả: `github.com/jakubkuzilek/oulad` (đọc `data/*.rda` bằng `pyreadr`), hoặc
- CSV chính thức: https://analyse.kmi.open.ac.uk/open-dataset

**Yêu cầu:** 7 file CSV gốc (`studentInfo.csv`, `studentRegistration.csv`, `studentAssessment.csv`, `assessments.csv`, `courses.csv`, `studentVle.csv`, `vle.csv`) phải nằm cùng thư mục với notebook.

> **Gợi ý tải nhanh từ R package bằng pyreadr:**
> ```python
> import pyreadr, pandas as pd
> # sau khi git clone github.com/jakubkuzilek/oulad
> mapping = {'student.rda':'studentInfo.csv','student_registration.rda':'studentRegistration.csv',
>            'student_assessment.rda':'studentAssessment.csv','assessment.rda':'assessments.csv',
>            'course.rda':'courses.csv','student_vle.rda':'studentVle.csv','vle.rda':'vle.csv'}
> for rda, csv in mapping.items():
>     df = list(pyreadr.read_r(f'oulad/data/{rda}').values())[0]
>     df.to_csv(csv, index=False)
> ```

<a id="p1-1"></a>
## 1.1 Tiền xử lý & Feature Engineering

Đọc 7 bảng, tổng hợp đặc trưng hành vi tại 3 mốc thời gian (25%, 50%, 75% thời lượng khóa học), xuất ra `master_{25,50,75}pct.csv`.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)

# ---- Load raw tables ----
studentInfo = pd.read_csv('studentInfo.csv')
studentReg = pd.read_csv('studentRegistration.csv')
studentAssess = pd.read_csv('studentAssessment.csv')
assessments = pd.read_csv('assessments.csv')
courses = pd.read_csv('courses.csv')
vle = pd.read_csv('studentVle.csv')  # Full official dataset (10,655,280 rows)
vle_meta = pd.read_csv('vle.csv')

print("=== RAW SHAPES ===")
for name, df in [('studentInfo', studentInfo), ('studentReg', studentReg),
                  ('studentAssess', studentAssess), ('assessments', assessments),
                  ('courses', courses), ('vle(studentVle)', vle), ('vle_meta', vle_meta)]:
    print(f"{name}: {df.shape}")

# ---- FULL DATASET: no scoping needed ----
# Verified official OULAD dataset (Kuzilek, Hlosta & Zdrahal, 2017), obtained from the
# original author's R package repository (jakubkuzilek/oulad), matching the published
# counts exactly: 32,593 students, 22 module-presentations, 10,655,280 VLE interactions.
studentAssess_full = studentAssess.copy()

print("\n=== FULL DATASET SHAPES (all 7 modules) ===")
print("studentInfo:", studentInfo.shape)
print("studentReg:", studentReg.shape)
print("assessments:", assessments.shape)
print("courses:", courses.shape)
print("vle interactions:", vle.shape)

# ---- Target variable ----
print("\n=== final_result distribution (scoped) ===")
print(studentInfo.final_result.value_counts())

# ---- Feature Engineering: behavioral features from VLE clickstream ----
# module_presentation_length needed to compute % thresholds
courses_len = courses.set_index(['code_module', 'code_presentation'])['module_presentation_length'].to_dict()

def pct_day(row_module, row_pres, day):
    length = courses_len.get((row_module, row_pres), 269)
    return day <= 0.25 * length, day <= 0.50 * length, day <= 0.75 * length

vle = vle.merge(courses[['code_module', 'code_presentation', 'module_presentation_length']],
                 on=['code_module', 'code_presentation'], how='left')
vle['length'] = vle['module_presentation_length'].fillna(269)
vle['pct_of_course'] = vle['date'] / vle['length']

def agg_clicks(cutoff):
    sub = vle[vle['pct_of_course'] <= cutoff]
    g = sub.groupby(['code_module', 'code_presentation', 'id_student']).agg(
        total_clicks=('sum_click', 'sum'),
        active_days=('date', 'nunique'),
        n_materials=('id_site', 'nunique'),
        first_access_day=('date', 'min'),
        last_access_day=('date', 'max'),
    ).reset_index()
    g.columns = ['code_module', 'code_presentation', 'id_student'] + \
                [f"{c}_{int(cutoff*100)}" for c in g.columns[3:]]
    return g

feat_25 = agg_clicks(0.25)
feat_50 = agg_clicks(0.50)
feat_75 = agg_clicks(0.75)

# ---- Assessment-based features (use only assessments due before each cutoff, scoped) ----
sa = studentAssess_full.merge(assessments, on='id_assessment', how='inner')
sa = sa.merge(courses[['code_module', 'code_presentation', 'module_presentation_length']],
              on=['code_module', 'code_presentation'], how='left')
sa['length'] = sa['module_presentation_length'].fillna(269)
sa['pct_of_course'] = sa['date_submitted'] / sa['length']

def agg_assess(cutoff):
    sub = sa[sa['pct_of_course'] <= cutoff]
    g = sub.groupby(['code_module', 'code_presentation', 'id_student']).agg(
        n_submitted=('id_assessment', 'count'),
        avg_score=('score', 'mean'),
        n_banked=('is_banked', 'sum'),
    ).reset_index()
    g.columns = ['code_module', 'code_presentation', 'id_student'] + \
                [f"{c}_{int(cutoff*100)}" for c in g.columns[3:]]
    return g

assess_25 = agg_assess(0.25)
assess_50 = agg_assess(0.50)
assess_75 = agg_assess(0.75)

# ---- Merge everything into one master table per cutoff ----
def build_dataset(cutoff_feat, cutoff_assess, label):
    df = studentInfo.merge(studentReg[['code_module', 'code_presentation', 'id_student',
                                        'date_registration']],
                            on=['code_module', 'code_presentation', 'id_student'], how='left')
    df = df.merge(cutoff_feat, on=['code_module', 'code_presentation', 'id_student'], how='left')
    df = df.merge(cutoff_assess, on=['code_module', 'code_presentation', 'id_student'], how='left')
    df['dataset_cutoff'] = label
    return df

df25 = build_dataset(feat_25, assess_25, '25pct')
df50 = build_dataset(feat_50, assess_50, '50pct')
df75 = build_dataset(feat_75, assess_75, '75pct')

for name, d in [('25%', df25), ('50%', df50), ('75%', df75)]:
    d.to_csv(f'master_{name.replace("%","pct")}.csv', index=False)
    print(f"\nSaved master dataset at {name} cutoff: {d.shape}")

print("\nPreprocessing complete.")


<a id="p1-2"></a>
## 1.2 Phân tích khám phá dữ liệu (EDA)

Tạo biểu đồ tổng quan (`eda_overview.png`) và bảng thống kê hành vi theo nhóm kết quả.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('master_75pct.csv')  # use full-course view for demographic EDA
print(df.shape)
print(df.final_result.value_counts())

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Final result distribution
order = df.final_result.value_counts().index
sns.countplot(data=df, x='final_result', order=order, ax=axes[0,0], palette='viridis')
axes[0,0].set_title('Phân bố kết quả học tập cuối cùng (n=7,044)')
axes[0,0].set_xlabel(''); axes[0,0].set_ylabel('Số sinh viên')

# 2. Result by gender
sns.countplot(data=df, x='gender', hue='final_result', ax=axes[0,1], palette='viridis')
axes[0,1].set_title('Kết quả học tập theo giới tính')
axes[0,1].set_xlabel('Giới tính'); axes[0,1].set_ylabel('Số sinh viên')

# 3. Result by highest_education
edu_order = df.highest_education.value_counts().index
sns.countplot(data=df, y='highest_education', hue='final_result', ax=axes[1,0],
              order=edu_order, palette='viridis')
axes[1,0].set_title('Kết quả học tập theo trình độ học vấn trước đó')
axes[1,0].set_xlabel('Số sinh viên'); axes[1,0].set_ylabel('')

# 4. Total clicks (75%) by result
sns.boxplot(data=df, x='final_result', y='total_clicks_75', order=order, ax=axes[1,1], palette='viridis')
axes[1,1].set_title('Tổng lượt click VLE (đến mốc 75%) theo kết quả')
axes[1,1].set_ylabel('Tổng lượt click'); axes[1,1].set_xlabel('')
axes[1,1].set_yscale('log')

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
print("Saved eda_overview.png")

# Summary stats table for the paper
summary = df.groupby('final_result')[['total_clicks_75', 'active_days_75', 'avg_score_75']].mean().round(1)
print("\n=== Mean behavioral features by outcome (75% cutoff) ===")
print(summary)
summary.to_csv('summary_by_outcome.csv')

# Correlation of numeric behavioral features with outcome (encode pass-ish vs not)
df['pass_like'] = df.final_result.isin(['Pass', 'Distinction']).astype(int)
num_cols = ['total_clicks_75', 'active_days_75', 'n_materials_75', 'avg_score_75', 'n_submitted_75']
corr = df[num_cols + ['pass_like']].corr()['pass_like'].drop('pass_like').sort_values(ascending=False)
print("\n=== Correlation with Pass/Distinction outcome ===")
print(corr)


<a id="p1-3"></a>
## 1.3 Huấn luyện & đánh giá 4 mô hình × 3 mốc thời gian

Huấn luyện Logistic Regression, Random Forest, SVM, XGBoost tại mỗi mốc (25/50/75%), cân bằng lớp bằng SMOTE, lưu kết quả vào `model_results_all_cutoffs.json`.

> **Lưu ý:** SVM giới hạn tập huấn luyện ở `SVM_MAX_TRAIN = 8000` để chạy khả thi trên máy 1 CPU. Tăng giá trị này nếu chạy trên máy mạnh hơn.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)
from sklearn.preprocessing import LabelEncoder, label_binarize
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import json
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

CAT_COLS = ['gender', 'region', 'highest_education', 'imd_band', 'age_band',
            'disability', 'code_module', 'code_presentation']
NUM_COLS = ['num_of_prev_attempts', 'studied_credits', 'date_registration',
            'total_clicks', 'active_days', 'n_materials', 'first_access_day',
            'last_access_day', 'n_submitted', 'avg_score', 'n_banked']

results_all = {}

for cutoff in ['25', '50', '75']:
    print(f"\n{'='*60}\nCUTOFF = {cutoff}%\n{'='*60}")
    df = pd.read_csv(f'master_{cutoff}pct.csv')

    # rename cutoff-specific cols to generic names
    rename_map = {}
    for c in df.columns:
        if c.endswith(f'_{cutoff}'):
            rename_map[c] = c[: -(len(cutoff) + 1)]
    df = df.rename(columns=rename_map)

    df = df[df.final_result != 'Withdrawn'].copy()  # focus: Fail/Pass/Distinction for clarity
    # (Withdrawn kept out of the 3-class task here; a separate binary at-risk task
    #  including Withdrawn is reported via the "pass_like" analysis in EDA)

    for c in NUM_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df[NUM_COLS] = df[NUM_COLS].fillna(0)
    for c in CAT_COLS:
        df[c] = df[c].fillna('Unknown').astype(str)

    X = df[CAT_COLS + NUM_COLS]
    y = df['final_result']

    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    classes = le.classes_
    print("Classes:", classes, "| distribution:", dict(zip(*np.unique(y, return_counts=True))))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=0.25, random_state=42, stratify=y_enc)

    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
        ('num', StandardScaler(), NUM_COLS),
    ])

    X_train_t = preprocessor.fit_transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    # SMOTE to balance classes in training set only
    sm = SMOTE(random_state=42)
    X_train_bal, y_train_bal = sm.fit_resample(X_train_t, y_train)

    models = {
        'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        'SVM': SVC(probability=True, random_state=42, cache_size=1000),
        'XGBoost': xgb.XGBClassifier(n_estimators=300, random_state=42, eval_metric='mlogloss',
                                      use_label_encoder=False, n_jobs=-1),
    }

    # SVM training cost scales poorly (O(n^2)-O(n^3)) with probability=True (internal CV).
    # On the full OULAD dataset the balanced training set is too large for tractable SVM
    # training, so we cap SVM's training subsample while keeping all other models on 100%
    # of the (SMOTE-balanced) training data.
    SVM_MAX_TRAIN = 8000

    cutoff_results = {}
    for name, model in models.items():
        print(f"  -> training {name} ...", flush=True)
        if name == 'SVM' and X_train_bal.shape[0] > SVM_MAX_TRAIN:
            rng = np.random.RandomState(42)
            idx = rng.choice(X_train_bal.shape[0], SVM_MAX_TRAIN, replace=False)
            model.fit(X_train_bal[idx], y_train_bal[idx])
        else:
            model.fit(X_train_bal, y_train_bal)
        y_pred = model.predict(X_test_t)
        y_proba = model.predict_proba(X_test_t)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        try:
            y_test_bin = label_binarize(y_test, classes=np.unique(y_enc))
            auc = roc_auc_score(y_test_bin, y_proba, average='weighted', multi_class='ovr')
        except Exception as e:
            auc = np.nan

        cutoff_results[name] = {
            'accuracy': round(acc, 4), 'precision': round(prec, 4),
            'recall': round(rec, 4), 'f1': round(f1, 4), 'auc_roc': round(auc, 4) if not np.isnan(auc) else None
        }
        print(f"{name:20s} | Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f} AUC={auc:.4f}")

    results_all[cutoff] = cutoff_results

    # Save best model artifacts at 75% for SHAP later
    if cutoff == '75':
        import joblib
        joblib.dump(models['XGBoost'], 'best_model_xgb_75.pkl')
        joblib.dump(preprocessor, 'preprocessor_75.pkl')
        np.save('X_test_75.npy', X_test_t if not hasattr(X_test_t, 'toarray') else X_test_t.toarray())
        X_test.to_csv('X_test_75_raw.csv', index=False)
        joblib.dump(le, 'label_encoder_75.pkl')
        # feature names
        feat_names = preprocessor.get_feature_names_out()
        with open('feature_names_75.json', 'w') as f:
            json.dump(list(feat_names), f)

with open('model_results_all_cutoffs.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print("\n\n=== SUMMARY TABLE (best F1 per cutoff) ===")
for cutoff, res in results_all.items():
    best = max(res.items(), key=lambda kv: kv[1]['f1'])
    print(f"{cutoff}%: best model = {best[0]}, F1={best[1]['f1']}, Acc={best[1]['accuracy']}, AUC={best[1]['auc_roc']}")


<a id="p1-4"></a>
## 1.4 Phân tích SHAP

Giải thích mô hình XGBoost tốt nhất (mốc 75%), xuất `shap_importance.png` và bảng top đặc trưng.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import json
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

model = joblib.load('best_model_xgb_75.pkl')
X_test = np.load('X_test_75.npy')
with open('feature_names_75.json') as f:
    feat_names = json.load(f)
le = joblib.load('label_encoder_75.pkl')

# Clean up feature names for readability
clean_names = [n.replace('cat__', '').replace('num__', '') for n in feat_names]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("SHAP values type:", type(shap_values), "shape:", np.array(shap_values).shape if not isinstance(shap_values, list) else [s.shape for s in shap_values])

# multiclass -> shap_values shape (n_samples, n_features, n_classes) for newer shap, or list of arrays for older
X_test_df = pd.DataFrame(X_test, columns=clean_names)

if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0).mean(axis=0)
elif shap_values.ndim == 3:
    mean_abs = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs = np.abs(shap_values).mean(axis=0)

importance = pd.Series(mean_abs, index=clean_names).sort_values(ascending=False)
print("\n=== TOP 15 FEATURES (mean |SHAP value|) ===")
print(importance.head(15))
importance.head(20).to_csv('shap_top20.csv')

# Plot
plt.figure(figsize=(9, 7))
top15 = importance.head(15).sort_values()
plt.barh(top15.index, top15.values, color='#4C72B0')
plt.xlabel('Mean |SHAP value| (mức độ ảnh hưởng trung bình)')
plt.title('Top 15 đặc trưng quan trọng nhất — Mô hình XGBoost (mốc 75%)')
plt.tight_layout()
plt.savefig('shap_importance.png', bbox_inches='tight', dpi=120)
print("\nSaved shap_importance.png")


<a id="p1-5"></a>
## 1.5 Biểu đồ bổ sung

Biểu đồ xu hướng dự đoán sớm (`early_prediction_trend.png`) và ma trận nhầm lẫn (`confusion_matrix_75.png`).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import confusion_matrix
import seaborn as sns

with open('model_results_all_cutoffs.json') as f:
    results = json.load(f)

# --- Chart 1: Early prediction trend ---
cutoffs = ['25', '50', '75']
models_order = ['LogisticRegression', 'RandomForest', 'SVM', 'XGBoost']
plt.figure(figsize=(8, 5.5))
for m in models_order:
    accs = [results[c][m]['f1'] for c in cutoffs]
    plt.plot(['25%', '50%', '75%'], accs, marker='o', linewidth=2, label=m)
plt.xlabel('Mốc thời gian trong khóa học (% thời lượng đã qua)')
plt.ylabel('F1-score (weighted)')
plt.title('Hiệu suất dự đoán theo mốc thời gian sớm (Early Prediction)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('early_prediction_trend.png', dpi=120, bbox_inches='tight')
print("Saved early_prediction_trend.png")

# --- Chart 2: Confusion matrix for best model (XGBoost @ 75%) ---
model = joblib.load('best_model_xgb_75.pkl')
X_test = np.load('X_test_75.npy')
le = joblib.load('label_encoder_75.pkl')

# need y_test - reload from raw pipeline quickly using same split logic
df = pd.read_csv('master_75pct.csv')
rename_map = {c: c[:-3] for c in df.columns if c.endswith('_75')}
df = df.rename(columns=rename_map)
df = df[df.final_result != 'Withdrawn'].copy()

from sklearn.model_selection import train_test_split
CAT_COLS = ['gender', 'region', 'highest_education', 'imd_band', 'age_band',
            'disability', 'code_module', 'code_presentation']
NUM_COLS = ['num_of_prev_attempts', 'studied_credits', 'date_registration',
            'total_clicks', 'active_days', 'n_materials', 'first_access_day',
            'last_access_day', 'n_submitted', 'avg_score', 'n_banked']
for c in NUM_COLS:
    if c not in df.columns: df[c] = np.nan
df[NUM_COLS] = df[NUM_COLS].fillna(0)
for c in CAT_COLS: df[c] = df[c].fillna('Unknown').astype(str)
y_enc = le.transform(df['final_result'])
X = df[CAT_COLS + NUM_COLS]
_, _, y_train, y_test = train_test_split(X, y_enc, test_size=0.25, random_state=42, stratify=y_enc)

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn — XGBoost (mốc 75%)')
plt.tight_layout()
plt.savefig('confusion_matrix_75.png', dpi=120, bbox_inches='tight')
print("Saved confusion_matrix_75.png")

print("\ny_test distribution:", np.unique(y_test, return_counts=True))
print("Accuracy check:", (y_pred == y_test).mean())


<a id="part2"></a>
# Phần 2 — Đối chứng #1: Bồ Đào Nha (2021)

**Dataset:** "Predict Students' Dropout and Academic Success" (Realinho et al., 2021), UCI ML Repository, `doi: 10.24432/C5MC89`.

**Yêu cầu:** file `dataset.csv` (4.424 sinh viên) trong cùng thư mục. Tải qua `pip install ucimlrepo` hoặc CSV mirror công khai.

Toàn bộ pipeline (tiền xử lý → 4 mô hình × 2 mốc học kỳ → SHAP) nằm trong một ô duy nhất.

In [ ]:
"""
Cross-dataset validation pipeline: UCI "Predict Students' Dropout and Academic
Success" (Realinho et al., 2021) -- a more recent (2021), different-institution,
different-country dataset used to validate whether the early-prediction
methodology developed on OULAD (2013-2014, UK) generalizes.

Two natural checkpoints (mirroring OULAD's 25/50/75% early-prediction design):
  - Checkpoint 1: after 1st semester only (demographics + 1st-sem curricular data)
  - Checkpoint 2: after 2nd semester (demographics + full-year curricular data)
"""
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('dataset.csv')
df.columns = [c.strip().lstrip('\ufeff') for c in df.columns]
print("Shape:", df.shape)
print(df['Target'].value_counts())

DEMOGRAPHIC_COLS = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Daytime/evening attendance', 'Previous qualification', 'Nacionality',
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", 'Displaced', 'Educational special needs', 'Debtor',
    'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment',
    'International', 'Unemployment rate', 'Inflation rate', 'GDP',
]
SEM1_COLS = [c for c in df.columns if '1st sem' in c]
SEM2_COLS = [c for c in df.columns if '2nd sem' in c]

CHECKPOINTS = {
    'sem1': DEMOGRAPHIC_COLS + SEM1_COLS,
    'sem2': DEMOGRAPHIC_COLS + SEM1_COLS + SEM2_COLS,
}

le = LabelEncoder()
y_all = le.fit_transform(df['Target'])
print("Classes:", le.classes_)

results_all = {}
saved = {}

for cp_name, feat_cols in CHECKPOINTS.items():
    print(f"\n{'='*60}\nCHECKPOINT = {cp_name} ({len(feat_cols)} features)\n{'='*60}")
    X = df[feat_cols].copy()
    y = y_all

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_t = scaler.fit_transform(X_train)
    X_test_t = scaler.transform(X_test)

    sm = SMOTE(random_state=42)
    X_train_bal, y_train_bal = sm.fit_resample(X_train_t, y_train)

    models = {
        'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        'SVM': SVC(probability=True, random_state=42, cache_size=1000),
        'XGBoost': xgb.XGBClassifier(n_estimators=300, random_state=42, eval_metric='mlogloss',
                                      use_label_encoder=False, n_jobs=-1),
    }

    cp_results = {}
    for name, model in models.items():
        print(f"  -> training {name} ...", flush=True)
        model.fit(X_train_bal, y_train_bal)
        y_pred = model.predict(X_test_t)
        y_proba = model.predict_proba(X_test_t)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        y_test_bin = label_binarize(y_test, classes=np.unique(y_all))
        auc = roc_auc_score(y_test_bin, y_proba, average='weighted', multi_class='ovr')

        cp_results[name] = {'accuracy': round(acc, 4), 'precision': round(prec, 4),
                             'recall': round(rec, 4), 'f1': round(f1, 4), 'auc_roc': round(auc, 4)}
        print(f"{name:20s} | Acc={acc:.4f} F1={f1:.4f} AUC={auc:.4f}")

        if cp_name == 'sem2' and name == 'XGBoost':
            joblib.dump(model, 'best_model_xgb_sem2.pkl')
            joblib.dump(scaler, 'scaler_sem2.pkl')
            joblib.dump(le, 'label_encoder.pkl')
            np.save('X_test_sem2.npy', X_test_t)
            with open('feature_names_sem2.json', 'w') as f:
                json.dump(feat_cols, f)
            cm = confusion_matrix(y_test, y_pred)
            np.save('confusion_matrix_sem2.npy', cm)

    results_all[cp_name] = cp_results

with open('model_results_dropout2021.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print("\n=== SUMMARY ===")
for cp, res in results_all.items():
    best = max(res.items(), key=lambda kv: kv[1]['f1'])
    print(f"{cp}: best = {best[0]}, F1={best[1]['f1']}, Acc={best[1]['accuracy']}, AUC={best[1]['auc_roc']}")

# ---- Early-prediction trend chart ----
plt.figure(figsize=(7.5, 5.3))
labels_map = {'sem1': 'Sau HK1', 'sem2': 'Sau HK2 (đầy đủ)'}
for m in ['LogisticRegression', 'RandomForest', 'SVM', 'XGBoost']:
    vals = [results_all[cp][m]['f1'] for cp in ['sem1', 'sem2']]
    plt.plot([labels_map['sem1'], labels_map['sem2']], vals, marker='o', linewidth=2, label=m)
plt.xlabel('Mốc thời gian (Dataset 2021)')
plt.ylabel('F1-score (weighted)')
plt.title('Hiệu suất dự đoán sớm — Dataset đối chứng 2021 (Bồ Đào Nha)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dropout2021_early_prediction_trend.png', dpi=120, bbox_inches='tight')
print("Saved dropout2021_early_prediction_trend.png")

# ---- Confusion matrix chart ----
cm = np.load('confusion_matrix_sem2.npy')
plt.figure(figsize=(6, 5.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn — XGBoost, HK2 đầy đủ (Dataset 2021)')
plt.tight_layout()
plt.savefig('dropout2021_confusion_matrix.png', dpi=120, bbox_inches='tight')
print("Saved dropout2021_confusion_matrix.png")

# ---- SHAP ----
model = joblib.load('best_model_xgb_sem2.pkl')
X_test = np.load('X_test_sem2.npy')
with open('feature_names_sem2.json') as f:
    feat_names = json.load(f)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0).mean(axis=0)
elif shap_values.ndim == 3:
    mean_abs = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs = np.abs(shap_values).mean(axis=0)

importance = pd.Series(mean_abs, index=feat_names).sort_values(ascending=False)
print("\n=== TOP 15 FEATURES (Dataset 2021) ===")
print(importance.head(15))
importance.head(20).to_csv('shap_top20_dropout2021.csv')

plt.figure(figsize=(9, 7))
top15 = importance.head(15).sort_values()
plt.barh(top15.index, top15.values, color='#2C5F2D')
plt.xlabel('Mean |SHAP value|')
plt.title('Top 15 đặc trưng quan trọng nhất — Dataset 2021 (XGBoost, HK2)')
plt.tight_layout()
plt.savefig('dropout2021_shap_importance.png', dpi=120, bbox_inches='tight')
print("Saved dropout2021_shap_importance.png")


<a id="part3"></a>
# Phần 3 — Đối chứng #2: Bắc Mỹ (2020)

**Dataset:** Injadat, Moubayed, Nassif & Shami (2020), *Applied Intelligence*. 486 sinh viên, 2 mốc thời gian (20%/50% tiến trình khóa học).

**Nguồn:** `github.com/Western-OC2-Lab/Student-Performance-and-Engagement-Prediction-eLearning-datasets` (folder "Student Performance Prediction - Multiclass Case").

**Yêu cầu:** file `dataset.csv` trong cùng thư mục.

> **Lưu ý:** dataset rất nhỏ (486 mẫu) và mất cân bằng cực đoan (chỉ 8 mẫu lớp Weak). Script tự điều chỉnh `k_neighbors` của SMOTE theo cỡ lớp thiểu số nhỏ nhất.

In [ ]:
"""
Third validation dataset: North American university course (Injadat et al., 2020,
Applied Intelligence) -- adds a third independent geographic/institutional context
(UK-OULAD 2013-14, Portugal 2021, North America ~2020) to the early-prediction
methodology, using the dataset's native checkpoints: 20% and 50% of coursework.
"""
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('dataset.csv')
print("Shape:", df.shape)
print(df['Class'].value_counts())

# ---- Checkpoints mirroring the dataset's native design (20% / 50% of coursework) ----
CHECKPOINTS = {
    'pct20': ['Quiz01 [10]', 'Assignment01 [8]'],                                   # ~20% of course
    'pct50': ['Quiz01 [10]', 'Assignment01 [8]', 'Midterm Exam [20]', 'Assignment02 [12]'],  # ~50%
}

le = LabelEncoder()
y_all = le.fit_transform(df['Class'])
print("Classes:", le.classes_, "counts:", np.bincount(y_all))

results_all = {}

for cp_name, feat_cols in CHECKPOINTS.items():
    print(f"\n{'='*60}\nCHECKPOINT = {cp_name} ({feat_cols})\n{'='*60}")
    X = df[feat_cols].copy()
    y = y_all

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_t = scaler.fit_transform(X_train)
    X_test_t = scaler.transform(X_test)

    # SMOTE: k_neighbors must be < smallest class count in training data
    min_class_count = pd.Series(y_train).value_counts().min()
    k_neighbors = max(1, min(5, min_class_count - 1))
    sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_bal, y_train_bal = sm.fit_resample(X_train_t, y_train)

    models = {
        'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        'SVM': SVC(probability=True, random_state=42, cache_size=1000),
        'XGBoost': xgb.XGBClassifier(n_estimators=300, random_state=42, eval_metric='mlogloss',
                                      use_label_encoder=False, n_jobs=-1),
    }

    cp_results = {}
    for name, model in models.items():
        print(f"  -> training {name} ...", flush=True)
        model.fit(X_train_bal, y_train_bal)
        y_pred = model.predict(X_test_t)
        y_proba = model.predict_proba(X_test_t)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        try:
            y_test_bin = label_binarize(y_test, classes=np.unique(y_all))
            auc = roc_auc_score(y_test_bin, y_proba, average='weighted', multi_class='ovr')
        except Exception:
            auc = np.nan

        cp_results[name] = {'accuracy': round(acc, 4), 'precision': round(prec, 4),
                             'recall': round(rec, 4), 'f1': round(f1, 4),
                             'auc_roc': round(auc, 4) if not np.isnan(auc) else None}
        print(f"{name:20s} | Acc={acc:.4f} F1={f1:.4f} AUC={auc:.4f}")

        if cp_name == 'pct50' and name == 'XGBoost':
            joblib.dump(model, 'best_model_xgb_pct50.pkl')
            joblib.dump(scaler, 'scaler_pct50.pkl')
            joblib.dump(le, 'label_encoder.pkl')
            np.save('X_test_pct50.npy', X_test_t)
            with open('feature_names_pct50.json', 'w') as f:
                json.dump(feat_cols, f)
            cm = confusion_matrix(y_test, y_pred)
            np.save('confusion_matrix_pct50.npy', cm)

    results_all[cp_name] = cp_results

with open('model_results_namerica.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print("\n=== SUMMARY ===")
for cp, res in results_all.items():
    best = max(res.items(), key=lambda kv: kv[1]['f1'])
    print(f"{cp}: best = {best[0]}, F1={best[1]['f1']}, Acc={best[1]['accuracy']}, AUC={best[1]['auc_roc']}")

# ---- Early-prediction trend chart ----
plt.figure(figsize=(7.5, 5.3))
labels_map = {'pct20': 'Mốc 20%', 'pct50': 'Mốc 50%'}
for m in ['LogisticRegression', 'RandomForest', 'SVM', 'XGBoost']:
    vals = [results_all[cp][m]['f1'] for cp in ['pct20', 'pct50']]
    plt.plot([labels_map['pct20'], labels_map['pct50']], vals, marker='o', linewidth=2, label=m)
plt.xlabel('Mốc thời gian (Dataset Bắc Mỹ)')
plt.ylabel('F1-score (weighted)')
plt.title('Hiệu suất dự đoán sớm — Dataset đối chứng Bắc Mỹ')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('namerica_early_prediction_trend.png', dpi=120, bbox_inches='tight')
print("Saved namerica_early_prediction_trend.png")

# ---- Confusion matrix chart ----
cm = np.load('confusion_matrix_pct50.npy')
plt.figure(figsize=(6, 5.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn — XGBoost, mốc 50% (Dataset Bắc Mỹ)')
plt.tight_layout()
plt.savefig('namerica_confusion_matrix.png', dpi=120, bbox_inches='tight')
print("Saved namerica_confusion_matrix.png")

# ---- SHAP ----
model = joblib.load('best_model_xgb_pct50.pkl')
X_test = np.load('X_test_pct50.npy')
with open('feature_names_pct50.json') as f:
    feat_names = json.load(f)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0).mean(axis=0)
elif shap_values.ndim == 3:
    mean_abs = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs = np.abs(shap_values).mean(axis=0)

importance = pd.Series(mean_abs, index=feat_names).sort_values(ascending=False)
print("\n=== FEATURES (Dataset Bắc Mỹ, mốc 50%) ===")
print(importance)
importance.to_csv('shap_namerica.csv')

plt.figure(figsize=(8, 4.5))
top = importance.sort_values()
plt.barh(top.index, top.values, color='#6A4C93')
plt.xlabel('Mean |SHAP value|')
plt.title('Đặc trưng quan trọng nhất — Dataset Bắc Mỹ (XGBoost, mốc 50%)')
plt.tight_layout()
plt.savefig('namerica_shap_importance.png', dpi=120, bbox_inches='tight')
print("Saved namerica_shap_importance.png")


<a id="part4"></a>
# Phần 4 — Đối chứng #3: Trung Đông (xAPI-Edu)

**Dataset:** xAPI-Edu-Data (Amrieh, Hamtini & Aljarah, 2016), thu thập từ LMS Kalboard 360 qua Experience API. 480 sinh viên, 3 lớp (L/M/H).

**Nguồn:** Kaggle `aljarah/xAPI-Edu-Data`. **Yêu cầu:** file `dataset.csv` (đổi tên từ `xAPI-Edu-Data.csv`) trong cùng thư mục.

**Đặc điểm quan trọng:** đây là dataset đối chứng có bản chất gần OULAD nhất — chứa 4 đặc trưng hành vi tương tác trực tuyến thuần túy (raisedhands, VisITedResources, AnnouncementsView, Discussion). Pipeline kiểm chứng kịch bản "chỉ hành vi" (không dùng điểm số) so với "đầy đủ".

In [ ]:
"""
Fourth validation dataset: xAPI-Edu-Data (Kalboard 360 LMS, Amman/Kuwait,
collected 2016; Amrieh, Hamtini & Aljarah). This dataset is the closest in SPIRIT
to OULAD among all our validation sets: it contains genuine online-interaction
behavioral features (raisedhands, VisITedResources, AnnouncementsView, Discussion)
rather than only exam/curricular scores -- directly analogous to OULAD's clickstream.

It adds a 4th independent geographic context (UK, Portugal, N.America, Middle East)
and lets us test the "behavior-only early prediction" idea in its purest form:
  - Checkpoint 1 (behavior-only): only the 4 interaction features -- the OULAD-style
    "can we predict from engagement behavior alone?" question.
  - Checkpoint 2 (full): behavior features + demographic/contextual features.
"""
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('dataset.csv')
print("Shape:", df.shape)
print(df['Class'].value_counts())

BEHAVIOR_COLS = ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']
CONTEXT_CAT = ['gender', 'NationalITy', 'PlaceofBirth', 'StageID', 'GradeID',
               'SectionID', 'Topic', 'Semester', 'Relation', 'ParentAnsweringSurvey',
               'ParentschoolSatisfaction', 'StudentAbsenceDays']

# Two checkpoints
CHECKPOINTS = {
    'behavior_only': (BEHAVIOR_COLS, []),
    'full': (BEHAVIOR_COLS, CONTEXT_CAT),
}

le = LabelEncoder()
# Order classes L < M < H for interpretability
df['Class'] = pd.Categorical(df['Class'], categories=['L', 'M', 'H'], ordered=True)
y_all = le.fit_transform(df['Class'].astype(str))
print("Classes:", le.classes_, "counts:", np.bincount(y_all))

results_all = {}

for cp_name, (num_cols, cat_cols) in CHECKPOINTS.items():
    print(f"\n{'='*60}\nCHECKPOINT = {cp_name}\n{'='*60}")
    X = df[num_cols + cat_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype(str)
    y = y_all

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y)

    if cat_cols:
        pre = ColumnTransformer([
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ])
    else:
        pre = ColumnTransformer([('num', StandardScaler(), num_cols)])

    X_train_t = pre.fit_transform(X_train)
    X_test_t = pre.transform(X_test)
    if hasattr(X_train_t, 'toarray'):
        X_train_t = X_train_t.toarray(); X_test_t = X_test_t.toarray()

    sm = SMOTE(random_state=42)
    X_train_bal, y_train_bal = sm.fit_resample(X_train_t, y_train)

    models = {
        'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        'SVM': SVC(probability=True, random_state=42, cache_size=1000),
        'XGBoost': xgb.XGBClassifier(n_estimators=300, random_state=42, eval_metric='mlogloss',
                                      use_label_encoder=False, n_jobs=-1),
    }

    cp_results = {}
    for name, model in models.items():
        print(f"  -> training {name} ...", flush=True)
        model.fit(X_train_bal, y_train_bal)
        y_pred = model.predict(X_test_t)
        y_proba = model.predict_proba(X_test_t)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        y_test_bin = label_binarize(y_test, classes=np.unique(y_all))
        auc = roc_auc_score(y_test_bin, y_proba, average='weighted', multi_class='ovr')

        cp_results[name] = {'accuracy': round(acc, 4), 'precision': round(prec, 4),
                             'recall': round(rec, 4), 'f1': round(f1, 4), 'auc_roc': round(auc, 4)}
        print(f"{name:20s} | Acc={acc:.4f} F1={f1:.4f} AUC={auc:.4f}")

        if cp_name == 'full' and name == 'XGBoost':
            joblib.dump(model, 'best_model_xgb_full.pkl')
            joblib.dump(pre, 'preprocessor_full.pkl')
            joblib.dump(le, 'label_encoder.pkl')
            np.save('X_test_full.npy', X_test_t)
            feat_names = list(pre.get_feature_names_out())
            with open('feature_names_full.json', 'w') as f:
                json.dump(feat_names, f)
            cm = confusion_matrix(y_test, y_pred)
            np.save('confusion_matrix_full.npy', cm)

    results_all[cp_name] = cp_results

with open('model_results_xapi.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print("\n=== SUMMARY ===")
for cp, res in results_all.items():
    best = max(res.items(), key=lambda kv: kv[1]['f1'])
    print(f"{cp}: best = {best[0]}, F1={best[1]['f1']}, Acc={best[1]['accuracy']}, AUC={best[1]['auc_roc']}")

# ---- Trend chart ----
plt.figure(figsize=(7.5, 5.3))
labels_map = {'behavior_only': 'Chỉ hành vi\ntương tác', 'full': 'Hành vi +\nbối cảnh đầy đủ'}
for m in ['LogisticRegression', 'RandomForest', 'SVM', 'XGBoost']:
    vals = [results_all[cp][m]['f1'] for cp in ['behavior_only', 'full']]
    plt.plot([labels_map['behavior_only'], labels_map['full']], vals, marker='o', linewidth=2, label=m)
plt.xlabel('Bộ đặc trưng (Dataset xAPI-Edu, Trung Đông)')
plt.ylabel('F1-score (weighted)')
plt.title('Hiệu suất dự đoán — Dataset đối chứng xAPI-Edu')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('xapi_early_prediction_trend.png', dpi=120, bbox_inches='tight')
print("Saved xapi_early_prediction_trend.png")

# ---- Confusion matrix ----
cm = np.load('confusion_matrix_full.npy')
plt.figure(figsize=(6, 5.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn — XGBoost, đầy đủ (Dataset xAPI-Edu)')
plt.tight_layout()
plt.savefig('xapi_confusion_matrix.png', dpi=120, bbox_inches='tight')
print("Saved xapi_confusion_matrix.png")

# ---- SHAP ----
model = joblib.load('best_model_xgb_full.pkl')
X_test = np.load('X_test_full.npy')
with open('feature_names_full.json') as f:
    feat_names = json.load(f)
clean = [n.replace('num__', '').replace('cat__', '') for n in feat_names]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0).mean(axis=0)
elif shap_values.ndim == 3:
    mean_abs = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs = np.abs(shap_values).mean(axis=0)

importance = pd.Series(mean_abs, index=clean).sort_values(ascending=False)
print("\n=== TOP 15 FEATURES (xAPI-Edu) ===")
print(importance.head(15))
importance.head(20).to_csv('shap_xapi.csv')

plt.figure(figsize=(9, 6))
top = importance.head(12).sort_values()
plt.barh(top.index, top.values, color='#D35400')
plt.xlabel('Mean |SHAP value|')
plt.title('Top 12 đặc trưng quan trọng nhất — Dataset xAPI-Edu (XGBoost)')
plt.tight_layout()
plt.savefig('xapi_shap_importance.png', dpi=120, bbox_inches='tight')
print("Saved xapi_shap_importance.png")


---

## Ghi chú cuối

- Mỗi phần độc lập nhau; có thể chạy riêng miễn là file dữ liệu tương ứng (`dataset.csv` hoặc 7 file OULAD) nằm đúng thư mục làm việc.
- Các file đầu ra (biểu đồ `.png`, kết quả `.json`, model `.pkl`) được lưu vào thư mục làm việc hiện tại.
- Để tái tạo chính xác kết quả trong bài báo, chạy lần lượt Phần 1 → 4.